##  **Pre-processing Core Idea :**

Each `.pickle` file consists two very different things together — thousands of **mel-spectrograms** (the actual sound) and **metadata** (who sang it, what song, what style). Cramming both into one CSV means writing raw audio numbers as *text* — bloating a single file to several gigabytes for nothing.

So we split them:

| **mel-spectrograms** | **metadata** |
|---|---|
| `{style}_melspec.npy` | `{style}.csv` |
| Binary, compact, fast to load | song · genre · artist · gender |
| Shape `(N, 128, 130)` | One row per segment |


# 1. Imports

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
OUTPUT_ROOT = "/kaggle/working/folk_music_converted"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# 2. Helper Functions

In [3]:
def decode_bytes(arr):
    """converts byte string -> normal python string"""
    arr_py = np.array([x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in arr])
    return arr_py

In [4]:
def convert_pickle(pickle_path: str, style_name: str = None, output_root: str = OUTPUT_ROOT):

    # will automatically detect the Folk-Style
    if style_name is None:
        style_name = os.path.splitext(os.path.basename(pickle_path))[0]
    os.makedirs(output_root, exist_ok=True)

    # loading the pickle file
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        with open(pickle_path, "rb") as f:
            data = pickle.load(f)

    # --- convert the mel-spectrograms ---
    # ONE single array of shape (N, 128, 130)
    mel_spec = np.stack([
        spec.astype(np.float32)
        for spec in tqdm(data["mel_spec"], desc=f"{style_name} segments", unit="seg")
    ])

    # metadata
    song = decode_bytes(data["song"])
    genre = decode_bytes(data["genre"])
    artist = decode_bytes(data["artist"])
    gender = decode_bytes(data["gender"])

    n = len(mel_spec)
    lengths = {"mel_spec": n, "song": len(song), "genre": len(genre),
               "artist": len(artist), "gender": len(gender)}
    if len(set(lengths.values())) != 1:  
        print(f"  [warning] {style_name}: column length mismatch {lengths}")

    safe_name = style_name.replace(" ", "_")

    # save mel-spec as .npy file
    npy_path = os.path.join(output_root, f"{safe_name}_melspec.npy")
    np.save(npy_path, mel_spec)

    # table
    meta_df = pd.DataFrame({
        "index": np.arange(n),
        "song": song,
        "genre": genre,
        "artist": artist,
        "gender": gender,
    })

    # metadata table to csv file
    csv_path = os.path.join(output_root, f"{safe_name}.csv")
    meta_df.to_csv(csv_path, index=False)

    size_mb = os.path.getsize(npy_path) / (1024 ** 2)
    print(f"  [ok] {style_name}: {n} segments -> {os.path.basename(csv_path)} "
          f"+ {os.path.basename(npy_path)} ({size_mb:.1f} MB)\n")

    return meta_df

# 3. **Usage :**

```python
convert_pickle("PICKLE_FILE PATH")
```

## 3.1 Uttarakhandi

In [5]:
convert_pickle("/kaggle/input/datasets/mishbhaul/folk-music-dataset/Uttarakhandi/Uttarakhandi/Uttarakhandi.pickle")

Uttarakhandi segments:   0%|          | 0/17872 [00:00<?, ?seg/s]

  [ok] Uttarakhandi: 17872 segments -> Uttarakhandi.csv + Uttarakhandi_melspec.npy (1134.5 MB)



,index,song,genre,artist,gender
0,0,Chhala Jan Chhali,Uttarakhandi,Harendra Parihar,Male
1,1,Chhala Jan Chhali,Uttarakhandi,Harendra Parihar,Male
2,2,Chhala Jan Chhali,Uttarakhandi,Harendra Parihar,Male
3,3,Chhala Jan Chhali,Uttarakhandi,Harendra Parihar,Male
4,4,Chhala Jan Chhali,Uttarakhandi,Harendra Parihar,Male
...,...,...,...,...,...
17867,17867,Dharyaali Mund Topli Kaali,Uttarakhandi,Suresh Kala,Male
17868,17868,Dharyaali Mund Topli Kaali,Uttarakhandi,Suresh Kala,Male
17869,17869,Dharyaali Mund Topli Kaali,Uttarakhandi,Suresh Kala,Male
17870,17870,Dharyaali Mund Topli Kaali,Uttarakhandi,Suresh Kala,Male


## 3.2 Veeragase

In [6]:
convert_pickle("/kaggle/input/datasets/mishbhaul/folk-music-dataset/Veeragase/Veeragase/Veeragase.pickle")

Veeragase segments:   0%|          | 0/15458 [00:00<?, ?seg/s]

  [ok] Veeragase: 15458 segments -> Veeragase.csv + Veeragase_melspec.npy (981.2 MB)



,index,song,genre,artist,gender
0,0,Veeragase Kunitakke,Veeragase,Ajay Warrior,Male
1,1,Veeragase Kunitakke,Veeragase,Ajay Warrior,Male
2,2,Veeragase Kunitakke,Veeragase,Ajay Warrior,Male
3,3,Veeragase Kunitakke,Veeragase,Ajay Warrior,Male
4,4,Veeragase Kunitakke,Veeragase,Ajay Warrior,Male
...,...,...,...,...,...
15453,15453,Morabada Veerane Karune,Veeragase,Sneha,Female
15454,15454,Morabada Veerane Karune,Veeragase,Sneha,Female
15455,15455,Morabada Veerane Karune,Veeragase,Sneha,Female
15456,15456,Morabada Veerane Karune,Veeragase,Sneha,Female
